# Write Coelho 2025 Figure 1A as linked ATST

One readout retains all 24 OD600 curves: eight source columns each for PA14, PA14 + VAC1 and PA14 + VAC3.

In [1]:
from pathlib import Path
import datetime
import pandas as pd
from openpyxl import load_workbook
from ATST import ATSTFile, MultiReadoutATST, validate_atst, write_multi_readout_atst
from ATST.blocks import Assay, Entities, EntityTable, FileMetadata, Layout, Metadata, Readings, ReadoutIds, Study

root=Path.cwd()
sheet=load_workbook(root/"sources/42003_2024_7269_MOESM4_ESM.xlsx",read_only=True,data_only=True)["Figure 1"]

In [2]:
file_info=FileMetadata(data={"file_name":"atst_coelho_2025.atst.txt","format":"ATST","format_version":"0.1","created_on":datetime.date.today().isoformat(),"field_delimiter":"TAB","encoding":"UTF-8"})
study=Study(data={"title":"Mucosal-adapted bacteriophages as a preventive strategy for a lethal Pseudomonas aeruginosa challenge in mice","study_id":"doi:10.1038/s42003-024-07269-0","authors":"Luiz Felipe Leomil Coelho et al.","publication_date":"2025","publication_doi":"10.1038/s42003-024-07269-0","description":"Figure 1A OD600 trajectories for PA14 with no phage, VAC1 or VAC3; eight workbook columns per condition."})
entities=Entities(tables={name:EntityTable(name="ENTITIES",table_name=name,pk=pk,data=pd.read_csv(root/"entities_definitions"/f"{name}.csv",dtype=str,keep_default_na=False)) for name,pk in [("ISOLATES","ISOLATE_ID"),("PHAGES","PHAGE_ID"),("MEDIA","MEDIUM_ID")]})
assay=Assay(data={"readout_type":"absorbance","readout_unit":"od600","time_unit":"h","temperature_c":"37","shaking_speed_rpm":"120","vessel":"5 mL LB liquid culture","medium_id":"LB","phage_moi":"0.01 for VAC1/VAC3; 0 for control","inoculum":"100 uL PA14 suspension at 2x10^8 cells/mL in sterile 0.9% NaCl, added to 5 mL LB","measurement_method":"spectrophotometry at 600 nm","sampling_times_h":"0, 1, 2, 17, 19, 21, 24, 26, 40 (workbook)","source_curves_per_condition":"8","reported_replicates":"Figure 1 caption says experiments performed in three replicates; workbook contains eight columns per condition","source_reference":"doi:10.1038/s42003-024-07269-0"})
metadata=Metadata(data={"source_file":"read_shared_data/sources/42003_2024_7269_MOESM4_ESM.xlsx","source_header_row":"2","source_time_range":"A3:A11","source_control_range":"B3:I11","source_VAC1_range":"J3:Q11","source_VAC3_range":"R3:Y11","conversion":"Preserve all 24 numeric source columns and irregular time values; assign local curve IDs in left-to-right column order"})

## Read source columns into one layout and readings table

`curve_A` corresponds to Excel B and `curve_X` to Excel Y. Replicate 1-8 labels are local source-column order, not established biological replicate IDs.

In [3]:
assert sheet["A1"].value=="FIGURE 1A"
assert [sheet.cell(2,c).value for c in [2,10,18]]==["PA14","PA14+ VAC1","PA14+VAC3"]
times=[sheet.cell(row,1).value for row in range(3,12)]
readings=pd.DataFrame({"Time":times})
layout_rows=[]
for i in range(24):
    column=i+2
    phage="" if i<8 else "VAC1" if i<16 else "VAC3"
    cid=f"curve_{chr(65+i)}"
    values=[sheet.cell(row,column).value for row in range(3,12)]
    assert all(isinstance(v,(int,float)) for v in values)
    readings[cid]=values
    layout_rows.append({"curve_id":cid,"type":"uninfected_control" if not phage else "phage_exposed","isolate_id":"PA14","phage_id":phage,"moi":"0" if not phage else "0.01","replicate":str(i%8+1)})
layout=pd.DataFrame(layout_rows)
readout=ATSTFile(file_info=file_info,study=study,metadata=metadata,assay=assay,entities=entities,layout=Layout(data=layout),readings=Readings(data=readings))
readout_ids=ReadoutIds(data=pd.DataFrame({"readout_id":["Fig_1A"]}))
readout.readout_ids=readout_ids
bundle=MultiReadoutATST(file_info,study,readout_ids,{"Fig_1A":readout},entities)
validate_atst(bundle)
print(len(layout),"curves x",len(readings),"time points")

24 curves x 9 time points


In [4]:
output=write_multi_readout_atst(bundle,root.parent/"atst_coelho_2025",linked_files=True)
print(output)

/Users/juanmantilla/Documents/TKI/ATST/examples/Public data /Coelho 2025/atst_coelho_2025/atst_coelho_2025.atst.txt
